In [65]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

In [66]:
df = pd.read_csv("../../data/H2_cleaned.csv")

In [67]:
season_map = {
    'December': 'Winter', 'January': 'Winter', 'February': 'Winter',
    'March': 'Spring',    'April': 'Spring',   'May': 'Spring',
    'June': 'Summer',     'July': 'Summer',    'August': 'Summer',
    'September': 'Fall',  'October': 'Fall',   'November': 'Fall'
}
df['Season'] = df['ArrivalDateMonth'].str.strip().map(season_map)

In [68]:
cols_to_drop = [
    # 'IsCanceled', # post booking
    'ReservationStatus', # post booking
    'ReservationStatusDate', # post booking, makes no sense without ReservationStatus anyways
    'AssignedRoomType', # post booking
    'BookingChanges',
    'DaysInWaitingList',
    'Agent',
    'ArrivalDateYear', # temporal
    'ArrivalDateDayOfMonth', # temporal
    'ArrivalDateWeekNumber', # temporal
    'ArrivalDateMonth', # temporal
    # 'CustomerType', # high leakage risk (objective is to discover segments)
    'DepositType',
    'MarketSegment',
    # 'IsRepeatedGuest'
    'PreviousCancellations',
    'PreviousBookingsNotCancelled'
]

df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Remaining columns ({df.shape[1]}):", df.columns.tolist())

Remaining columns (19): ['IsCanceled', 'LeadTime', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'Meal', 'Country', 'DistributionChannel', 'IsRepeatedGuest', 'PreviousBookingsNotCanceled', 'ReservedRoomType', 'CustomerType', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests', 'HasCompany', 'Season']


In [69]:
cols_to_encode = [
    'Meal',
    'Country',
    'DistributionChannel',
    'ReservedRoomType',
    'Season',
    'CustomerType'
]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoder_transformer = ColumnTransformer(
    transformers=[
        (
            "nominal_encoded",
            encoder,
            cols_to_encode,
        )
    ],
    remainder="passthrough",
).set_output(transform="pandas")

df = encoder_transformer.fit_transform(df)


In [74]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

kmeans = KMeans(n_clusters=7, random_state=42)
kmeans.fit(scaled_data)

df['cluster'] = kmeans.labels_
cluster_labels = kmeans.fit_predict(scaled_data)

# evaluate
silhouette = silhouette_score(scaled_data, cluster_labels)
db_index = davies_bouldin_score(scaled_data, cluster_labels)

print("Silhouette Score:", silhouette)
print("Davies-Bouldin Index:", db_index)


Silhouette Score: 0.079266057818083
Davies-Bouldin Index: 2.6317397163647054


In [ ]:
# Scale the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

# Apply K-Means
kmeans = KMeans(n_clusters=7, random_state=42)

cluster_labels = kmeans.fit_predict(scaled_data)

# Add cluster labels to the original dataframe
df['cluster'] = cluster_labels

# Evaluate
silhouette = silhouette_score(scaled_data, cluster_labels)
db_index = davies_bouldin_score(scaled_data, cluster_labels)

print("Silhouette Score:", silhouette)
print("Davies-Bouldin Index:", db_index)

Silhouette Score: 0.08829690708192503
Davies-Bouldin Index: 2.4686089903966084


In [72]:
# # Scale the data
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(df)

# silhouette_scores = []
# db_scores = []
# k_values = range(2, 11)

# for k in k_values:

#     kmeans = KMeans(n_clusters=k, random_state=42)

#     labels = kmeans.fit_predict(X_scaled)

#     silhouette = silhouette_score(X_scaled, labels)
#     db_index = davies_bouldin_score(X_scaled, labels)

#     silhouette_scores.append(silhouette)
#     db_scores.append(db_index)

#     print(f"K = {k}")
#     print(f"Silhouette Score: {silhouette:.4f}")
#     print(f"Davies-Bouldin Index: {db_index:.4f}")
#     print("-------------------------")

In [73]:
# Z = linkage(df, method="ward")

# plt.figure(figsize=(10, 6))
# dendrogram(
#     Z,
#     truncate_mode="lastp",  # Truncate the tree if there are too many data points
#     p=30,  # Show only the last 30 merged clusters
#     leaf_rotation=90.0,
#     leaf_font_size=10.0,
# )

# plt.title("Hierarchical Clustering Dendrogram")
# plt.xlabel("Data Point Index / (Cluster Size)")
# plt.ylabel("Distance (Ward)")
# plt.axhline(y=15, color="r", linestyle="--")  # Visual cut line to choose cluster count
# plt.show()